# ViHSD Mixture of Experts experiment (Kaggle Edition)

This notebook prepares a **Kaggle Notebook runtime** to train and evaluate the ViHSD Mixture of Experts (MoE) architectures.

Training and evaluation are executed explicitly from shell commands. This keeps experiment execution reproducible, parameter overrides transparent, and run outputs systematically tracked by unique `run_id`s.

### Essential Kaggle Settings Checklist
Before running any cells, make sure these settings are enabled in the Kaggle right-hand **Notebook settings** sidebar:
1. **Accelerator**: Select **GPU P100** (single 16GB GPU) or **GPU T4 x2** (15GB VRAM per card).
2. **Internet**: Toggle **Internet ON** (requires SMS verification on Kaggle). Internet is required to clone this repository, download the ViHSD dataset from Hugging Face, download PhoBERT weights, and log to Weights & Biases.
3. **Secrets**: Under the notebook's top menu, open **Add-ons → Secrets** and create:
   - `HF_TOKEN`: Hugging Face access token.
   - `WANDB_API_KEY`: Weights & Biases API key.
4. **Output Persistence**: Everything written to `/kaggle/working` is saved permanently when you click **"Save Version" → "Save & Run All (Commit)"**. Completed runs will appear in the notebook's **Output** tab.

---

## 1. Configure Kaggle Output Directories

On Kaggle, `/kaggle/working` is the only writable directory during execution. We configure `CHECKPOINT_DIR` and `RESULTS_DIR` environment variables so all model checkpoints (`.safetensors`) and experiment logs (`run_metrics.json`, `vihsd_predictions.json`) are saved directly to `/kaggle/working/checkpoints` and `/kaggle/working/results`.

In [ ]:
import os
from pathlib import Path

checkpoint_dir = Path('/kaggle/working/checkpoints')
results_dir = Path('/kaggle/working/results')

checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

os.environ['CHECKPOINT_DIR'] = str(checkpoint_dir)
os.environ['RESULTS_DIR'] = str(results_dir)

print(f"CHECKPOINT_DIR: {os.environ['CHECKPOINT_DIR']}")
print(f"RESULTS_DIR:    {os.environ['RESULTS_DIR']}")

## 2. Clone the Repository & Verify Dependencies

This step ensures the repository code is available in the workspace and targets the `kaggle-version` branch.

If you are running this notebook inside an already cloned or uploaded workspace where `train.py` is present, it will automatically use the current directory without re-cloning.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = '/kaggle/working/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'
BRANCH = 'kaggle-version'

if not Path('train.py').exists():
    if not Path(PROJECT_DIR).exists():
        print(f"Cloning {REPOSITORY_URL} (branch: {BRANCH})...")
        !git clone --depth 1 --branch $BRANCH $REPOSITORY_URL $PROJECT_DIR
    %cd $PROJECT_DIR
else:
    print(f"Already in project repository root: {Path.cwd()}")

%pip install -q -r requirements.txt

## 3. Authenticate via Kaggle Secrets

Authentication credentials for Hugging Face and Weights & Biases are retrieved securely using Kaggle's `UserSecretsClient`.

Ensure you have added `HF_TOKEN` and `WANDB_API_KEY` under **Add-ons → Secrets** in the notebook toolbar.

In [ ]:
import os
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    wandb_api_key = user_secrets.get_secret('WANDB_API_KEY')
except Exception as error:
    print(f"Kaggle secrets client notice: {error}")
    hf_token = os.getenv('HF_TOKEN')
    wandb_api_key = os.getenv('WANDB_API_KEY')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("✓ Hugging Face authentication configured.")
else:
    print("! Notice: HF_TOKEN secret not found; public Hugging Face assets will still load.")

if wandb_api_key:
    os.environ['WANDB_API_KEY'] = wandb_api_key
    wandb.login(key=wandb_api_key)
    print("✓ Weights & Biases authentication successful.")
else:
    print("! Notice: WANDB_API_KEY secret not found; set logging.use_wandb=false if W&B is not needed.")

## 4. Run Training and Evaluation Manually

The command cells below execute the training and evaluation scripts. Each run saves its resolved configuration, checkpoint, and metrics to `CHECKPOINT_DIR/<run-id>` and `RESULTS_DIR/<run-id>`.

### Fast Smoke Test
Run a quick 1-epoch smoke test on a 2,000-sample subset to verify GPU access, dataset loading, and tokenization:
```bash
python train.py --config configs/vihsd.yaml --smoke-test --run-id smoke-check
python evaluate.py --config configs/vihsd.yaml --run-id smoke-check
```

### Full Baseline Experiment
Run the complete training profile with a unique `--run-id`:
```bash
python train.py --config configs/vihsd.yaml --no-smoke-test --run-id baseline-current-moe
python evaluate.py --config configs/vihsd.yaml --run-id baseline-current-moe
```

### Experiment with Architecture or Hyperparameter Overrides
Use `--set section.key=value` to override any parameter without editing `configs/vihsd.yaml`:
```bash
python train.py \
  --config configs/vihsd.yaml \
  --no-smoke-test \
  --run-id stronger-moe-focal \
  --set model.architecture=stronger_moe \
  --set model.num_experts=8 \
  --set model.top_k=2 \
  --set training.loss_type=focal \
  --set training.focal_gamma=2.0
python evaluate.py --config configs/vihsd.yaml --run-id stronger-moe-focal
```

## 5. Training Command

Edit the command below and execute the cell manually. For a persistent run, save a Kaggle version via **Save Version → Save & Run All (Commit)**.

In [ ]:
# Edit this command for your experiment, then run the cell.
!python train.py --config configs/vihsd.yaml --no-smoke-test --run-id baseline-current-moe

# Example variant:
# !python train.py --config configs/vihsd.yaml --no-smoke-test --run-id stronger-moe-v1 --set model.architecture=stronger_moe --set model.num_experts=8 --set model.top_k=2 --set training.loss_type=focal

## 6. Evaluation Command

Run evaluation after training finishes. Pass the matching `--run-id` to load that run's best checkpoint (`vihsd_moe_best.safetensors`) and compute test set metrics.

In [ ]:
# Use the run_id from the training command above:
!python evaluate.py --config configs/vihsd.yaml --run-id baseline-current-moe

# Or evaluate a specific checkpoint directly:
# !python evaluate.py --config configs/vihsd.yaml --checkpoint /kaggle/working/checkpoints/baseline-current-moe/vihsd_moe_best.safetensors

## 7. Command-Line Argument Reference

### Common `--set` Configuration Keys
Any existing key in `configs/vihsd.yaml` can be overridden using `--set section.key=value`:

| Section | Key | Description / Options |
| :--- | :--- | :--- |
| **Model** | `model.architecture` | `current_moe` / `stronger_moe` / `pretrained_backbone` |
| | `model.num_experts` | Total number of experts (default: 4) |
| | `model.top_k` | Active experts per token (`1 <= top_k <= num_experts`) |
| | `model.model_dim` | Internal model hidden dimension |
| | `model.expert_hidden_dim` | Feed-forward dimension inside each expert |
| | `model.num_layers` | Number of Transformer layers |
| | `model.dropout` | Dropout probability |
| **Training** | `training.epochs` | Number of full epochs |
| | `training.batch_size` | Batch size per step (default: 16) |
| | `training.learning_rate` | AdamW learning rate (default: 0.0002) |
| | `training.loss_type` | `cross_entropy`, `weighted_cross_entropy`, or `focal` |
| | `training.focal_gamma` | Gamma focusing parameter for focal loss |
| | `training.class_weights` | Class weight list, e.g. `[1.0, 1.5, 2.0]` |
| | `training.num_workers` | DataLoader workers; keep at `0` to prevent Kaggle `/dev/shm` bus errors |
| **Logging** | `logging.use_wandb` | `true` / `false` to toggle W&B syncing |
| **General** | `seed` | Random seed for dataset splitting & weights |

## 8. Summary of Evaluation Metrics

Inspect the test set performance for your run directly in the notebook.

In [ ]:
import json
import os
from pathlib import Path

# Set this to the run_id you want to inspect
RUN_ID = 'baseline-current-moe'
results_root = Path(os.getenv('RESULTS_DIR', '/kaggle/working/results'))
metrics_path = results_root / RUN_ID / 'run_metrics.json'

if not metrics_path.exists():
    print(f"No saved metrics found at: {metrics_path}")
else:
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(f"=== Evaluation Summary: {RUN_ID} ===")
    print(f"Best Validation Epoch: {metrics.get('best_epoch')}")
    test_data = metrics.get('test', {})
    display({
        'Test Loss': test_data.get('loss'),
        'Accuracy': test_data.get('accuracy'),
        'Macro F1': test_data.get('macro_f1'),
        'Weighted F1': test_data.get('weighted_f1'),
    })
    if 'per_class_f1' in test_data:
        print("Per-class F1 Scores:")
        for label, score in test_data['per_class_f1'].items():
            print(f"  {label}: {score:.4f}")